In [1]:
!apt install swig cmake ffmpeg xvfb python3-opengl

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.15).
Suggested packages:
  libgle3 python3-numpy python3-tk swig-doc swig-examples swig4.0-examples
  swig4.0-doc
The following NEW packages will be installed:
  freeglut3 libglu1-mesa python3-opengl swig swig4.0
0 upgraded, 5 newly installed, 0 to remove and 35 not upgraded.
Need to get 1,940 kB of archives.
After this operation, 13.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 freeglut3 amd64 2.8.1-6 [74.0 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglu1-mesa amd64 9.0.2-1 [145 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 python3-opengl all 3.1.5+dfsg-1 [605 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/u

In [2]:
import os

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

!pip install pyvirtualdisplay imageio[ffmpeg]

%env MUJOCO_GL=egl
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

env: MUJOCO_GL=egl


In [3]:
# Prepare to load data from google drive
from google.colab import drive
import datetime

# CONNECT TO GOOGLE DRIVE
gdrive_path = '/content/drive'
drive.mount(gdrive_path)

# DEFINE WORK DIRECTORY
current_step = 'step_005'
# workDir = f'{gdrive_path}/My Drive/Research/{current_step}'
workDir = os.path.join(gdrive_path, 'My Drive', 'Research', current_step)
print('WorkDir:', workDir)

log_dir = os.path.join(workDir, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
print('LogDir:', log_dir)

# create folder if it doesn't exists
if not os.path.exists(log_dir):
  os.makedirs(log_dir)

tf_log_dir = os.path.join(workDir, 'tf_logs')
print('TfLogDir:', tf_log_dir)

# create folder if it doesn't exists
if not os.path.exists(tf_log_dir):
  os.makedirs(tf_log_dir)

Mounted at /content/drive
WorkDir: /content/drive/My Drive/Research/step_005
LogDir: /content/drive/My Drive/Research/step_005/20250903-002636
TfLogDir: /content/drive/My Drive/Research/step_005/tf_logs


In [4]:
# clean content folder
import os
import shutil

# location
location = "/content"

# directories
dirs = ["sample_data", "rl-zoo", "gym_darwin_op3", "videos"]

for dir in dirs:
    path = os.path.join(location, dir)
    try:
        shutil.rmtree(path)
    except OSError as e:
        print("Error: %s : %s" % (path, e.strerror))

Error: /content/rl-zoo : No such file or directory
Error: /content/gym_darwin_op3 : No such file or directory
Error: /content/videos : No such file or directory


In [5]:
# Install Darwin Model
model_path = '/content/gym_darwin_op3'

if os.path.isdir(model_path):
  print(f"The directory '{model_path}' exists - git pull")
  %cd {model_path}
  !git pull
  %cd /
else:
  print(f"The directory '{model_path}' does not exist - git clone")
  !git clone --single-branch --branch {current_step} https://github.com/Gianzanti/robofei_mestrado.git {model_path}


The directory '/content/gym_darwin_op3' does not exist - git clone
Cloning into '/content/gym_darwin_op3'...
remote: Enumerating objects: 249, done.
remote: Counting objects: 100% (249/249), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 249 (delta 86), reused 173 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (249/249), 14.83 MiB | 30.87 MiB/s, done.
Resolving deltas: 100% (86/86), done.


In [6]:
!pip install -e {model_path}

Obtaining file:///content/gym_darwin_op3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 27.4 MB/s eta 0:00:00
  Building editable for robofei (pyproject.toml) ... done
  Created wheel for robofei: filename=robofei-0.1.6-py3-none-any.whl size=1438 sha256=8d7c6bf189a0b7a93efb034bdab543801da9fa33969e86cd210f0348b29ad44b
  Stored in directory: /tmp/pip-ephem-wheel-cache-f2gekzow/wheels/43/c3/48/682f53e738aa575222935107724428d2ce2d742b055ebb4adc
Successfully built robofei


In [7]:
# Install RL Zoo
trainner_path = '/content/rl-zoo'

if os.path.isdir(trainner_path):
  print(f"The directory '{trainner_path}' exists - git pull")
  %cd {trainner_path}
  !git pull
  %cd /
else:
  print(f"The directory '{trainner_path}' does not exist - git clone")
  !git clone https://github.com/Gianzanti/rl-zoo.git {trainner_path}


The directory '/content/rl-zoo' does not exist - git clone
Cloning into '/content/rl-zoo'...
remote: Enumerating objects: 3657, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 3657 (delta 35), reused 24 (delta 15), pack-reused 3597 (from 2)
Receiving objects: 100% (3657/3657), 8.55 MiB | 26.37 MiB/s, done.
Resolving deltas: 100% (2229/2229), done.


In [8]:
!pip install -e {trainner_path}

Obtaining file:///content/rl-zoo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.4/247.4 kB 29.8 MB/s eta 0:00:00
  Building editable for rl_zoo3 (pyproject.toml) ... done
  Created wheel for rl_zoo3: filename=rl_zoo3-2.7.0-0.editable-py3-none-any.whl size=4428 sha256=1ba9d4def77db0e3b6956424996d457c169e8f1814f5afcbfac18c60e3935497
  Stored in directory: /tmp/pip-ephem-wheel-cache-7fzfzcxl/wheels/2d/0b/3c/be4c09b7d9d2a891b5d1bc1e086a8fe4f4b4f016a0613b625c
Successfully built rl_zoo3


In [9]:
# Hyper Parameters Tunning
%cd {trainner_path}

algos_cpu = ['ppo', 'a2c']
algos_cuda = ['ddpg', 'sac', 'td3']

n_timestep = 5_000_000
save_freq = min(50_000, int(n_timestep / 10))
eval_freq = min(100_000, int(n_timestep / 10))

max_episode_steps = 1000
wrapper = [{"gymnasium.wrappers.TimeLimit": {"max_episode_steps": max_episode_steps}}]

for algo in algos_cpu:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
    ctrl_cost_weight:1e-3 target_distance:2.0 forward_velocity_weight:1.0 \
    --hyperparams n_timesteps:{n_timestep} env_wrapper:"{wrapper}"\
    --device cpu
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n 2000 \
    --load-best -o "{log_dir}" -f "{log_dir}"

for algo in algos_cuda:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
    ctrl_cost_weight:1e-3 target_distance:2.0 forward_velocity_weight:1.0 \
    --hyperparams n_timesteps:{n_timestep} env_wrapper:"{wrapper}"\
    --device cuda
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n 2000 \
    --load-best -o "{log_dir}" -f "{log_dir}"



Streaming output truncated to the last 5000 lines.
|    time_elapsed    | 6312     |
|    total_timesteps | 4843600  |
| train/             |          |
|    actor_loss      | -2.84    |
|    critic_loss     | 0.00817  |
|    learning_rate   | 0.001    |
|    n_updates       | 604199   |
---------------------------------
---------------------------------
| mean_episode/      |          |
|    control_cost    | 0.143    |
|    forward_reward  | 0.835    |
|    health_reward   | 0.5      |
|    pos_x           | 0.681    |
|    pos_y           | 0.187    |
|    pos_z           | 0.283    |
|    vel_x           | 0.835    |
|    vel_y           | 0.178    |
| rollout/           |          |
|    ep_len_mean     | 189      |
|    ep_rew_mean     | 279      |
| time/              |          |
|    episodes        | 36956    |
|    fps             | 767      |
|    time_elapsed    | 6313     |
|    total_timesteps | 4844432  |
| train/             |          |
|    actor_loss      | -2.81   